# N-Candidate Solver — Drawer Slides (N=1)

The simplest case for the `NCandidateSolver`: a **single-product family**. There is no Cartesian product — the solver evaluates each slide individually against the customer's requirements.

This demonstrates that the same solver algorithm handles N=1 (drawer slides), N=2 (concealed hinges — see `v2_hinge_n_candidate_demo.ipynb`), and N=3 (LED lighting — see `../v2-n-candidate/v2_n_candidate_demo.ipynb`) without any structural change.

## The Domain

A contractor needs drawer slides for a specific cabinet. They know:
- **Cabinet depth** — how deep the cabinet is (determines max slide length)
- **Drawer weight** — how much the loaded drawer weighs (determines load rating needed)
- **Preferences** — extension type, mount type, soft-close, push-open, disconnect

The engine finds all slides that satisfy the hard constraints and match the preferences.

### Why N=1?

Unlike hinges (hinge + plate) or LED lighting (bar + driver + dimmer), drawer slides are a standalone product. There is no second product to pair with. The `NCandidateSolver` handles this as a degenerate case of the Cartesian product: one role, one list, iterate and evaluate.

### Rules (8)

| Rule | Category | Constraint |
|---|---|---|
| DS001 load_capacity | Hard | Drawer weight ≤ slide max load |
| DS002 cabinet_depth | Hard | Cabinet deep enough for slide + clearance |
| DS003 extension_type | Hard | Matches requested extension (if specified) |
| DS004 mount_type | Hard | Matches requested mount type (if specified) |
| DS005 undermount_width | Hard | Drawer width ≤ 900mm for undermount (if applicable) |
| DS006 disconnect | Preference | Has tool-free disconnect (if required) |
| DS007 soft_close | Preference | Has soft-close damping (if requested) |
| DS008 push_open | Preference | Has push-open/touch-latch (if requested) |

**Data:** `sample-data/drawer_slides.json` (4 products, synthetic)

In [ ]:
import sys
from pathlib import Path

# Find project root (works regardless of kernel working directory)
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver
from engine_v2.families.drawer_slide.config import SLIDE_N_CONFIG
from engine_v2.families.drawer_slide.loader import load_from_json
from engine_v2.families.drawer_slide.models import (
    ExtensionType, SlideCloseType, SlideMountType, SlideRequirements,
)

all_slides = load_from_json(DATA_DIR)

solver = NCandidateSolver(
    config=SLIDE_N_CONFIG,
    product_lists={"slide": all_slides},
)

print(f"Loaded {len(all_slides)} drawer slides from sample-data/drawer_slides.json")
print(f"Roles: {SLIDE_N_CONFIG.role_names}  (N={len(SLIDE_N_CONFIG.roles)})")
print(f"Rules: {len(SLIDE_N_CONFIG.rules)}")

## 1. Product Catalog

In [ ]:
print("=" * 110)
print("DRAWER SLIDE CATALOG")
print("=" * 110)
print(f"{'SKU':<16} {'Brand':<8} {'Series':<26} {'Length':>6} {'Load':>6} {'Extension':<16} {'Mount':<14} {'Close':<12} {'Disc':>5} {'Price':>7}")
print("-" * 110)
for s in all_slides:
    disc = "Yes" if s.disconnect_feature else "No"
    price = f"${s.price_usd:.2f}" if s.price_usd else "N/A"
    print(f"{s.sku:<16} {s.brand:<8} {s.series:<26} {s.slide_length_mm:>4}mm {s.max_load_kg:>4.0f}kg {s.extension_type.value:<16} {s.mount_type.value:<14} {s.close_type.value:<12} {disc:>5} {price:>7}")

## 2. Solving Scenarios

### Scenario 1: Standard kitchen drawer

550mm cabinet, 15kg drawer, no special preferences.

In [ ]:
req = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)

result = solver.solve_with_explanation(req)
print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print(f"Recommended: {rec['candidates']['slide']['sku']}")
    price = f"${rec['total_price_usd']:.2f}" if rec['total_price_usd'] else "N/A"
    print(f"Price: {price}\n")
    
    print(f"Constraint trace ({len(rec['constraint_trace'])} rules):")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"  [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    
    print(f"\n+ {len(result['alternatives'])} alternative(s)")
    for alt in result["alternatives"]:
        alt_price = f"${alt['total_price_usd']:.2f}" if alt['total_price_usd'] else "N/A"
        print(f"  {alt['candidates']['slide']['sku']} — {alt_price}")

### Scenario 2: Heavy-duty — only the strongest slides

In [ ]:
req_heavy = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=42.0)
results = solver.solve(req_heavy)

print(f"Found {len(results)} slide(s) for 42kg drawer:\n")
for config in results:
    s = config.candidates["slide"]
    print(f"  {s.sku} — {s.brand} {s.series}, {s.max_load_kg}kg rating, ${s.price_usd:.2f}")

### Scenario 3: Specific preferences — full extension, undermount, soft-close, Blum only

In [ ]:
req_pref = SlideRequirements(
    cabinet_depth_mm=550,
    drawer_weight_kg=20.0,
    extension_type=ExtensionType.FULL,
    mount_type=SlideMountType.UNDERMOUNT,
    soft_close=True,
    preferred_brand="Blum",
)
results = solver.solve(req_pref)

print(f"Found {len(results)} slide(s) matching all preferences:\n")
for config in results:
    s = config.candidates["slide"]
    print(f"  {s.sku} — {s.brand} {s.series}")
    print(f"    Extension: {s.extension_type.value}, Mount: {s.mount_type.value}, Close: {s.close_type.value}")
    print(f"    Load: {s.max_load_kg}kg, Length: {s.slide_length_mm}mm, Price: ${s.price_usd:.2f}")

### Scenario 4: Impossible — cabinet too shallow

In [ ]:
req_fail = SlideRequirements(cabinet_depth_mm=300, drawer_weight_kg=5.0)
result = solver.solve_with_explanation(req_fail)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result.get("closest_match"):
    cm = result["closest_match"]
    print(f"Closest match: {cm['candidates']['slide']['sku']}")

print(f"\nFailed constraints:")
for fr in result["failed_rules"]:
    print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")
    if fr.get("remediation"):
        print(f"           Fix: {fr['remediation']}")

## 3. Exhaustive Evaluation

With N=1, "exhaustive evaluation" just means evaluating each slide against all 8 rules. No Cartesian product — the search space equals the catalog size.

In [ ]:
# Evaluate every slide with early termination off
solver.config.early_termination = False
baseline = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)

print(f"{'SKU':<16} {'Valid':>5}  Rule Results")
print("-" * 90)
for slide in all_slides:
    config = solver.evaluate({"slide": slide}, baseline)
    status = "YES" if config.valid else "NO"
    rules = "  ".join(
        f"{r.rule_id}:{'PASS' if r.passed else 'FAIL'}" for r in config.rule_results
    )
    print(f"{slide.sku:<16} {status:>5}  {rules}")

solver.config.early_termination = True
print(f"\nTotal evaluations: {len(all_slides)} slides x {len(SLIDE_N_CONFIG.rules)} rules = {len(all_slides) * len(SLIDE_N_CONFIG.rules)}")
print(f"For N=1 there is no Cartesian product — complexity is O(products x rules).")

## Summary

| Aspect | Drawer Slides (N=1) |
|---|---|
| **Roles** | slide |
| **Cartesian product** | None — iterate products |
| **Rules** | 8 (5 hard, 3 preferences) |
| **Catalog** | 4 slides (synthetic) |
| **Evaluations** | 4 × 8 = 32 |
| **Data source** | `sample-data/drawer_slides.json` |

The same `NCandidateSolver` that handles this single-product family also handles:
- **N=2** concealed hinges (53 × 55 = 2,915 pairs) — see `v2_hinge_n_candidate_demo.ipynb`
- **N=3** LED lighting (5 × 4 × 4 = 80 triples) — see `../v2-n-candidate/v2_n_candidate_demo.ipynb`

**Source code:** `engine_v2/core/solver_n.py` (solver), `engine_v2/families/drawer_slide/` (models, rules, loader)